<a href="https://colab.research.google.com/github/harshithauv20/About-Me/blob/main/trafficlights.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install if needed:
# pip install opencv-python numpy

import cv2
import numpy as np
import time
import threading
import random
from datetime import datetime
from collections import deque

print(" Libraries loaded")

 Libraries loaded


In [ ]:
# Screen size for each camera feed
FRAME_W = 640
FRAME_H = 480

PHASE_DURATIONS = {
    "NS_GREEN":  8,   # North-South green
    "NS_YELLOW": 2,   # North-South yellow
    "EW_GREEN":  8,   # East-West green
    "EW_YELLOW": 2,   # East-West yellow
}

PHASES = ["NS_GREEN", "NS_YELLOW", "EW_GREEN", "EW_YELLOW"]

SIGNAL_COLORS = {
    "NS_GREEN":  {"ns": "green",  "ew": "red"},
    "NS_YELLOW": {"ns": "yellow", "ew": "red"},
    "EW_GREEN":  {"ns": "red",    "ew": "green"},
    "EW_YELLOW": {"ns": "red",    "ew": "yellow"},
}

# ── BGR colors used for drawing ──
COLORS = {
    "red":    (0,   0,   220),
    "yellow": (0,   210, 255),
    "green":  (0,   200, 80),
    "off":    (30,  30,  30),
    "white":  (220, 220, 220),
    "dark":   (15,  15,  15),
}

print("Config ready")

Config ready


In [ ]:
class Vehicle:
    """
    Represents ONE moving car on a lane.
    - Spawns at the edge of the frame
    - Moves toward the other edge
    - Resets when it exits the frame
    """

    def __init__(self, lane: str):
        self.lane  = lane                                      # "north", "south", "east", "west"
        self.color = tuple(random.randint(60, 220) for _ in range(3))  # random car color
        self.speed = random.uniform(1.5, 3.5)                 # pixels per frame
        self.w     = random.randint(28, 42)                   # car width
        self.h     = random.randint(18, 28)                   # car height
        self._place_at_start()

    def _place_at_start(self):
        """Put car at the beginning of its lane."""
        if self.lane == "south":
            self.x = random.randint(FRAME_W // 3, 2 * FRAME_W // 3)
            self.y = -self.h                   # start above frame
        elif self.lane == "north":
            self.x = random.randint(FRAME_W // 3, 2 * FRAME_W // 3)
            self.y = FRAME_H + self.h          # start below frame
        elif self.lane == "east":
            self.y = random.randint(FRAME_H // 3, 2 * FRAME_H // 3)
            self.x = -self.w                   # start left of frame
        elif self.lane == "west":
            self.y = random.randint(FRAME_H // 3, 2 * FRAME_H // 3)
            self.x = FRAME_W + self.w          # start right of frame

    def move(self, green_light: bool):
        """Move car if light is green; stop if red."""
        if not green_light:
            return  # car waits at red

        if   self.lane == "south":  self.y += self.speed
        elif self.lane == "north":  self.y -= self.speed
        elif self.lane == "east":   self.x += self.speed
        elif self.lane == "west":   self.x -= self.speed

        # Reset if car exits the frame
        off_screen = (
            (self.lane == "south"  and self.y > FRAME_H + self.h) or
            (self.lane == "north"  and self.y < -self.h)          or
            (self.lane == "east"   and self.x > FRAME_W + self.w) or
            (self.lane == "west"   and self.x < -self.w)
        )
        if off_screen:
            self._place_at_start()

    def draw(self, frame):
        """Draw this car as a colored rectangle with headlights."""
        x1 = int(self.x - self.w / 2)
        y1 = int(self.y - self.h / 2)
        x2 = x1 + self.w
        y2 = y1 + self.h

        # Car body
        cv2.rectangle(frame, (x1, y1), (x2, y2), self.color, -1)
        # Car outline
        cv2.rectangle(frame, (x1, y1), (x2, y2), COLORS["white"], 1)
        # Headlights (two small yellow dots)
        cv2.circle(frame, (x1 + 5,  y1 + 4), 3, (200, 200, 100), -1)
        cv2.circle(frame, (x2 - 5,  y1 + 4), 3, (200, 200, 100), -1)

print(" Vehicle class ready")

 Vehicle class ready


In [ ]:
class LaneCamera:
    """
    Simulates ONE road camera watching ONE lane.
    - Draws road + moving vehicles
    - Detects vehicles using OpenCV contours (like YOLO bounding boxes)
    - Reports vehicle count and congestion density
    """

    def __init__(self, lane: str, num_vehicles: int = 5):
        self.lane     = lane
        self.vehicles = [Vehicle(lane) for _ in range(num_vehicles)]
        self.density  = 0.0   # congestion % (0–100)
        self.detected = 0     # number of vehicles found by detector
        self._stagger_positions()

    def _stagger_positions(self):
        """Spread vehicles so they don't all start at the same spot."""
        for i, v in enumerate(self.vehicles):
            gap = i * 95
            if   self.lane == "south": v.y = -v.h  - gap
            elif self.lane == "north": v.y = FRAME_H + gap
            elif self.lane == "east":  v.x = -v.w  - gap
            elif self.lane == "west":  v.x = FRAME_W + gap

    def update(self, green_light: bool):
        """Move all vehicles on this lane."""
        for v in self.vehicles:
            v.move(green_light)

    def get_frame(self) -> np.ndarray:
        """
        Returns one camera frame with:
        1. Road drawn
        2. Vehicles drawn
        3. Detection bounding boxes
        4. HUD info overlay
        """
        # ── Step 1: Black background ──
        frame = np.zeros((FRAME_H, FRAME_W, 3), dtype=np.uint8)

        # ── Step 2: Draw road (grey strip) ──
        if self.lane in ("north", "south"):
            cv2.rectangle(frame,
                          (FRAME_W // 3, 0),
                          (2 * FRAME_W // 3, FRAME_H),
                          (35, 35, 35), -1)
            # dashed centre line
            for y in range(0, FRAME_H, 30):
                cv2.line(frame, (FRAME_W // 2, y), (FRAME_W // 2, y + 15), (70, 70, 70), 2)
        else:
            cv2.rectangle(frame,
                          (0, FRAME_H // 3),
                          (FRAME_W, 2 * FRAME_H // 3),
                          (35, 35, 35), -1)
            for x in range(0, FRAME_W, 30):
                cv2.line(frame, (x, FRAME_H // 2), (x + 15, FRAME_H // 2), (70, 70, 70), 2)

        # ── Step 3: Draw each vehicle ──
        for v in self.vehicles:
            v.draw(frame)

        # ── Step 4: Detect vehicles using contours ──
        gray      = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        _, mask   = cv2.threshold(gray, 40, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        self.detected = 0
        for c in contours:
            if cv2.contourArea(c) > 400:                      # ignore tiny noise
                x, y, w, h = cv2.boundingRect(c)
                cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 100), 2)   # green bbox
                cv2.putText(frame, "VEH", (x, y - 4),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 255, 100), 1)
                self.detected += 1

        # ── Step 5: Compute density score ──
        self.density = min(100.0, self.detected * 14.0 + random.uniform(-4, 4))

        # ── Step 6: HUD text bar at top ──
        status = "CONGESTED" if self.density > 70 else "MODERATE" if self.density > 40 else "CLEAR"
        hud    = f"{self.lane.upper():6s} | {self.detected} VEH | {self.density:.0f}% | {status}"
        cv2.rectangle(frame, (0, 0), (FRAME_W, 22), (0, 0, 0), -1)
        cv2.putText(frame, hud, (6, 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 220, 120), 1)

        # ── Step 7: Timestamp ──
        ts = datetime.now().strftime("%H:%M:%S")
        cv2.putText(frame, f"REC {ts}", (FRAME_W - 110, 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.38, (60, 60, 200), 1)

        return frame

print("LaneCamera class ready")

LaneCamera class ready


In [ ]:
def draw_signal_panel(phase: str,
                      ns_density: float,
                      ew_density: float,
                      countdown: int,
                      log_lines: list) -> np.ndarray:
    """
    Draws the right-side control panel showing:
    - Two traffic lights (N/S and E/W)
    - Current phase name + countdown
    - Density bars
    - System log
    """
    panel = np.full((FRAME_H, 420, 3), 12, dtype=np.uint8)

    # ── Helper: draw one traffic light bulb ──
    def draw_bulb(cx, cy, color_name, is_active):
        color = COLORS[color_name] if is_active else COLORS["off"]
        cv2.circle(panel, (cx, cy), 22, color, -1)
        if is_active:
            cv2.circle(panel, (cx, cy), 28, tuple(c // 3 for c in color), 2)  # glow ring
        cv2.circle(panel, (cx, cy), 22, (60, 60, 60), 1)  # outline

    # ── Helper: draw full 3-bulb traffic light ──
    def draw_light(x_center, label, direction):
        active_color = SIGNAL_COLORS[phase][direction]

        # Housing box
        cv2.rectangle(panel, (x_center - 32, 58), (x_center + 32, 218), (28, 28, 28), -1)
        cv2.rectangle(panel, (x_center - 32, 58), (x_center + 32, 218), (55, 55, 55), 1)

        # Label above
        cv2.putText(panel, label, (x_center - 20, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 180, 100), 1)

        # Three bulbs: red (top), yellow (middle), green (bottom)
        draw_bulb(x_center, 95,  "red",    active_color == "red")
        draw_bulb(x_center, 145, "yellow", active_color == "yellow")
        draw_bulb(x_center, 195, "green",  active_color == "green")

    # Draw both traffic lights
    draw_light(105, "N / S", "ns")
    draw_light(315, "E / W", "ew")

    # ── Phase name ──
    phase_color = (0, 220, 100) if "GREEN" in phase else (0, 210, 255)
    cv2.putText(panel, phase.replace("_", " "), (10, 248),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, phase_color, 2)

    # ── Countdown timer ──
    cv2.putText(panel, f"{countdown}s", (340, 248),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 160), 2)

    # ── Density bars ──
    def draw_bar(y, label, value, bar_color):
        cv2.putText(panel, f"{label}: {value:.0f}%", (10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (120, 180, 120), 1)
        bar_width = int((value / 100) * 380)
        cv2.rectangle(panel, (10, y + 5), (390, y + 13), (30, 30, 30), -1)   # background
        cv2.rectangle(panel, (10, y + 5), (10 + bar_width, y + 13), bar_color, -1)  # fill

    draw_bar(272, "N/S DENSITY", ns_density, (0, 180, 80))
    draw_bar(292, "E/W DENSITY", ew_density, (180, 120, 0))

    # ── System log ──
    cv2.putText(panel, "── SYSTEM LOG ──", (10, 322),
                cv2.FONT_HERSHEY_SIMPLEX, 0.38, (0, 100, 60), 1)

    for i, line in enumerate(list(log_lines)[-7:]):
        brightness = max(40, 200 - i * 25)
        cv2.putText(panel, line, (10, 342 + i * 18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.32, (0, brightness, brightness // 2), 1)

    # ── Border ──
    cv2.rectangle(panel, (0, 0), (419, FRAME_H - 1), (0, 80, 40), 1)
    cv2.putText(panel, "NEXUS TRAFFIC AI v2.4", (10, 472),
                cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 60, 40), 1)

    return panel

print("Signal panel function ready")

Signal panel function ready


In [ ]:
class TrafficController:
    """
    The BRAIN of the system.
    - Manages phase switching with a timer thread
    - Reads density from all 4 cameras
    - Applies AI logic to shorten phases when one side is overloaded
    - Renders the final display window
    """

    def __init__(self):
        self.phase_idx  = 0
        self.countdown  = PHASE_DURATIONS[PHASES[0]]
        self.log        = deque(maxlen=20)
        self.running    = True

        # Create one camera per lane
        self.cameras = {
            "north": LaneCamera("north", num_vehicles=5),
            "south": LaneCamera("south", num_vehicles=5),
            "east":  LaneCamera("east",  num_vehicles=5),
            "west":  LaneCamera("west",  num_vehicles=5),
        }

        self._log("SYSTEM BOOT — Nexus Traffic AI")
        self._log("4 camera feeds initialized")
        self._log("Detection engine: ACTIVE")
        self._log("Adaptive AI: ENABLED")

    # ── Shortcut: current phase name ──
    @property
    def phase(self):
        return PHASES[self.phase_idx]

    # ── Add a timestamped log entry ──
    def _log(self, msg: str):
        ts = datetime.now().strftime("%H:%M:%S")
        entry = f"[{ts}] {msg}"
        self.log.append(entry)
        print(entry)

    # ── Move to the next phase ──
    def _next_phase(self):
        self.phase_idx = (self.phase_idx + 1) % len(PHASES)
        self.countdown = PHASE_DURATIONS[self.phase]
        self._log(f"Phase → {self.phase}")

    # ── AI: shorten overloaded phases ──
    def _ai_adjust(self, ns_density: float, ew_density: float):
        diff = ns_density - ew_density
        if abs(diff) > 35 and "YELLOW" not in self.phase:
            if diff > 0 and "EW" in self.phase:
                self.countdown = max(1, self.countdown - 2)
                self._log(f"AI: N/S heavy ({ns_density:.0f}%) — cutting E/W time")
            elif diff < 0 and "NS" in self.phase:
                self.countdown = max(1, self.countdown - 2)
                self._log(f"AI: E/W heavy ({ew_density:.0f}%) — cutting N/S time")

    # ── Background thread: counts down and switches phases ──
    def _phase_timer(self):
        while self.running:
            time.sleep(1)
            self.countdown -= 1
            if self.countdown <= 0:
                self._next_phase()

print("TrafficController class (logic) ready")

TrafficController class (logic) ready


In [ ]:
def run(controller: TrafficController):
    """
    Main loop:
    1. Update all cameras
    2. Render 4 camera frames into a 2x2 grid
    3. Render signal panel
    4. Combine and show in one window
    5. Handle keyboard input
    """

    # Import cv2_imshow for Colab compatibility
    from google.colab.patches import cv2_imshow

    # Start the phase countdown in background
    timer_thread = threading.Thread(target=controller._phase_timer, daemon=True)
    timer_thread.start()

    controller._log("Press Q=quit  SPACE=skip phase")

    frame_count = 0

    while controller.running:
        frame_count += 1
        sig = SIGNAL_COLORS[controller.phase]   # current signal state

        # ── Step 1: Update cameras based on signal ──
        for lane, cam in controller.cameras.items():
            if lane in ("north", "south"):
                green = (sig["ns"] == "green")
            else:
                green = (sig["ew"] == "green")
            cam.update(green_light=green)

        # ── Step 2: Get a rendered frame from each camera ──
        f_north = controller.cameras["north"].get_frame()
        f_south = controller.cameras["south"].get_frame()
        f_east  = controller.cameras["east"].get_frame()
        f_west  = controller.cameras["west"].get_frame()

        # ── Step 3: Scale each to half size (320x240) ──
        def half(f): return cv2.resize(f, (320, 240))
        top    = np.hstack([half(f_north), half(f_south)])   # top row
        bottom = np.hstack([half(f_east),  half(f_west)])    # bottom row
        grid   = np.vstack([top, bottom])                    # 640 x 480 combined

        # ── Step 4: Compute average density per axis ──
        ns_density = (controller.cameras["north"].density +
                      controller.cameras["south"].density) / 2
        ew_density = (controller.cameras["east"].density +
                      controller.cameras["west"].density) / 2

        # ── Step 5: AI check every 30 frames ──
        if frame_count % 30 == 0:
            controller._ai_adjust(ns_density, ew_density)

        # ── Step 6: Draw signal panel ──
        panel = draw_signal_panel(
            controller.phase,
            ns_density,
            ew_density,
            controller.countdown,
            list(controller.log)
        )

        # ── Step 7: Combine grid + panel side by side ──
        display = np.hstack([grid, panel])   # final 1060 x 480 window

        # ── Step 8: Show window ──
        cv2_imshow(display)

        # ── Step 9: Keyboard input (wait 33ms ≈ 30fps) ──
        key = cv2.waitKey(33) & 0xFF
        if key == ord("q"):
            controller._log("Shutdown by operator.")
            controller.running = False
        elif key == ord(" "):
            controller._log("Manual phase skip.")
            controller.countdown = 0

    cv2.destroyAllWindows()

print("Run loop ready")

Run loop ready


In [ ]:
# Create the controller
ctrl = TrafficController()

# Start the system
run(ctrl)